# arg-position-back-functions — worked example 1: Write sub_back0 and sub_back1 for out = x - y

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `arg-position-back-functions`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

For a multi-arg forward op you write **one backward per positional input**: `_back0` returns the gradient w.r.t. arg 0 (`x`), `_back1` returns the gradient w.r.t. arg 1 (`y`). For subtraction `out = x - y` the two are *almost* the same but differ by a sign — a small reminder that the per-arg back fns are generally not identical. Each back fn has the uniform signature `(grad_out, out, x, y)` and returns a tensor shaped like the arg it differentiates.

## Worked solution

**Goal.** Implement the two per-argument back fns for `out = x - y`.

**Step 1 — local derivatives.** Differentiate the forward elementwise. `d(x - y)/dx = +1` and `d(x - y)/dy = -1`. These are the *local* gradients of the output w.r.t. each input.

**Step 2 — chain rule.** Upstream gradient is `grad_out = dL/dout`. By the chain rule `dL/dx = grad_out * (dout/dx) = grad_out * 1 = grad_out`, and `dL/dy = grad_out * (dout/dy) = grad_out * (-1) = -grad_out`.

**Step 3 — write the back fns.** `sub_back0` just returns `grad_out` unchanged; `sub_back1` returns `-grad_out`. Note we never need `out`, `x`, or `y` here — but the signature still carries them so every back fn in the registry is callable the same way.

**Step 4 — why the sign split matters.** This is the smallest example of asymmetry: even though both bodies are one line, registering the *same* function at both arg positions would silently flip the sign of `dL/dy`. The convention forces you to think about each position independently.

In [ ]:
def sub_back0(grad_out: Tensor, out: Tensor, x: Tensor, y: Tensor) -> Tensor:
    # d(x - y)/dx = +1  ->  dL/dx = grad_out
    return grad_out


def sub_back1(grad_out: Tensor, out: Tensor, x: Tensor, y: Tensor) -> Tensor:
    # d(x - y)/dy = -1  ->  dL/dy = -grad_out
    return -grad_out


t.manual_seed(0)
x = t.randn(2, 3, requires_grad=True)
y = t.randn(2, 3, requires_grad=True)
out = x - y
grad_out = t.randn(2, 3)
out.backward(grad_out)

gx = sub_back0(grad_out, out.detach(), x.detach(), y.detach())
gy = sub_back1(grad_out, out.detach(), x.detach(), y.detach())
print('grad_x match:', t.allclose(gx, x.grad))
print('grad_y match:', t.allclose(gy, y.grad))